# Stock Price Trend Analysis of NVIDIA (NVDA) Using Python
**ITM University Gwalior – PBL Mini Project | Data Analytics Using Python (CSL-0312)**

Dataset: `NVDA_Historical_Data.csv` (daily Open, High, Low, Close, Volume; 2020-01-02 to 2026-07-30).
Place the CSV in the same folder as this notebook (in Google Colab, upload it or mount Drive and change `CSV`).

## 1. Import libraries

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
%matplotlib inline

## 2. Load data and cleaning
The yfinance export has two extra header rows (Ticker, Date) that are skipped while loading.

In [ ]:
CSV = "NVDA_Historical_Data.csv"
df = pd.read_csv(CSV, skiprows=[1, 2]).rename(columns={"Price": "Date"})
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()

print("Shape:", df.shape)
print("Period:", df.index.min().date(), "to", df.index.max().date())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Duplicate dates:", int(df.index.duplicated().sum()))
bad = ((df.High < df.Low) | (df.Close > df.High + 1e-9) | (df.Close < df.Low - 1e-9)
       | (df.Open > df.High + 1e-9) | (df.Open < df.Low - 1e-9))
print("Inconsistent price rows:", int(bad.sum()))
df.head()

## 3. Feature engineering

In [ ]:
df["Return"] = df.Close.pct_change() * 100
df["LogRet"] = np.log(df.Close).diff() * 100
df["MA20"] = df.Close.rolling(20).mean()
df["MA50"] = df.Close.rolling(50).mean()
df["MA200"] = df.Close.rolling(200).mean()
df["Vol30"] = df.LogRet.rolling(30).std() * np.sqrt(252)   # annualised 30-day volatility
df["VolMA50"] = df.Volume.rolling(50).mean()
df["DD"] = (df.Close / df.Close.cummax() - 1) * 100         # drawdown from previous peak
df["Year"] = df.index.year
df["Weekday"] = df.index.day_name()

ret = df.Return.dropna()
q1, q3 = ret.quantile([.25, .75]); iqr = q3 - q1
print("IQR outliers in returns:", int(((ret < q1 - 1.5*iqr) | (ret > q3 + 1.5*iqr)).sum()))
print("Returns beyond 3 SD:", int((np.abs((ret - ret.mean()) / ret.std()) > 3).sum()))

## 4. Exploratory Data Analysis

In [ ]:
cols = ["Open", "High", "Low", "Close", "Volume", "Return"]
summary = df[cols].describe().T[["mean", "50%", "std", "min", "max"]]
summary["skew"] = df[cols].skew()
summary["kurtosis"] = df[cols].kurt()
summary.round(2)

In [ ]:
rows = []
for yr, g in df.groupby("Year"):
    prev = df.Close[df.index < g.index[0]]
    base = prev.iloc[-1] if len(prev) else g.Close.iloc[0]
    rows.append(dict(Year=yr, Days=len(g), StartClose=g.Close.iloc[0], EndClose=g.Close.iloc[-1],
                     ReturnPct=(g.Close.iloc[-1]/base - 1)*100,
                     AnnVolPct=g.LogRet.std()*np.sqrt(252),
                     AvgVolumeM=g.Volume.mean()/1e6,
                     MaxDDPct=((g.Close/g.Close.cummax() - 1)*100).min()))
yearly = pd.DataFrame(rows).set_index("Year")
yearly.round(2)

In [ ]:
years = (df.index[-1] - df.index[0]).days / 365.25
print(f"Total return      : {(df.Close.iloc[-1]/df.Close.iloc[0]-1)*100:.0f}%")
print(f"CAGR              : {((df.Close.iloc[-1]/df.Close.iloc[0])**(1/years)-1)*100:.1f}% per year")
print(f"Annual volatility : {df.LogRet.std()*np.sqrt(252):.1f}%")
print(f"Sharpe (rf = 0)   : {ret.mean()/ret.std()*np.sqrt(252):.2f}")
print(f"Max drawdown      : {df.DD.min():.1f}% on {df.DD.idxmin().date()}")
print(f"Up days           : {(ret > 0).mean()*100:.1f}%")
print("Best days :", [(str(d.date()), round(v, 2)) for d, v in ret.nlargest(3).items()])
print("Worst days:", [(str(d.date()), round(v, 2)) for d, v in ret.nsmallest(3).items()])

## 5. Data visualization

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(df.index, df.Close, lw=1, label="Close")
plt.plot(df.index, df.MA50, lw=1.2, label="50-day MA", color="orange")
plt.plot(df.index, df.MA200, lw=1.2, label="200-day MA", color="green")
plt.title("NVDA Closing Price with 50-day and 200-day Moving Averages")
plt.ylabel("Price (USD, adjusted)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.hist(ret, bins=70, density=True, edgecolor="black", alpha=.85)
x = np.linspace(ret.min(), ret.max(), 300)
plt.plot(x, stats.norm.pdf(x, ret.mean(), ret.std()), "r--", label="Normal fit")
plt.title("Distribution of Daily Returns"); plt.xlabel("Daily return (%)"); plt.ylabel("Density")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4.2))
sns.boxplot(data=df.dropna(subset=["Return"]), x="Year", y="Return", color="#9ecae1", fliersize=2)
plt.title("Daily Returns by Year"); plt.ylabel("Daily return (%)"); plt.tight_layout(); plt.show()

In [ ]:
cm = df[["Open", "High", "Low", "Close", "Volume", "Return"]].corr()
plt.figure(figsize=(6.2, 4.8))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="viridis", annot_kws={"size": 8})
plt.title("Correlation Heatmap of Numeric Variables"); plt.yticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(df.index, df.Vol30, color="crimson", lw=1.2)
plt.title("30-day Rolling Annualised Volatility"); plt.ylabel("Volatility (%)"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.fill_between(df.index, df.Volume/1e6, 0, color="#6baed6", alpha=.6, lw=0, label="Daily volume")
plt.plot(df.index, df.VolMA50/1e6, color="darkblue", lw=1.3, label="50-day average")
plt.title("Trading Volume Over Time"); plt.ylabel("Volume (millions)"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.fill_between(df.index, df.DD, 0, color="indianred", alpha=.6)
plt.title("Drawdown from Previous Peak"); plt.ylabel("Drawdown (%)"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
colors = ["seagreen" if v >= 0 else "indianred" for v in yearly.ReturnPct]
plt.bar(yearly.index.astype(str), yearly.ReturnPct, color=colors, edgecolor="black")
for i, v in enumerate(yearly.ReturnPct):
    plt.text(i, v + (8 if v >= 0 else -30), f"{v:.0f}%", ha="center", fontsize=8)
plt.title("Calendar-year Return (last year is year-to-date)"); plt.ylabel("Return (%)")
plt.tight_layout(); plt.show()

In [ ]:
t_idx = np.arange(len(df))
lr = stats.linregress(t_idx, np.log(df.Close))
plt.figure(figsize=(8, 4.2))
plt.semilogy(df.index, df.Close, lw=1, label="Close (log scale)")
plt.semilogy(df.index, np.exp(lr.intercept + lr.slope * t_idx), "r--",
             label=f"Log-linear trend (R² = {lr.rvalue**2:.3f})")
plt.title("Log-scale Price with Fitted Exponential Trend"); plt.ylabel("Price (USD, log scale)")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Statistical analysis

### 6.a Trend, correlation and regression

In [ ]:
r, p = stats.pearsonr(t_idx, df.Close)
print(f"Pearson r (time, Close): r = {r:.3f}, p = {p:.2e}")
lin = stats.linregress(t_idx, df.Close)
print(f"Linear regression    : R² = {lin.rvalue**2:.3f}, slope = {lin.slope:.3f} USD/day")
print(f"Log-linear regression: R² = {lr.rvalue**2:.3f}, ~{(np.exp(lr.slope*252)-1)*100:.1f}% growth per year")

vc = stats.pearsonr(df.Volume, df.Close)
print(f"Close vs Volume      : r = {vc[0]:.3f}, p = {vc[1]:.2e}, r² = {vc[0]**2:.3f}")
va = stats.pearsonr(df.Volume.iloc[1:], df.Return.abs().iloc[1:])
print(f"Volume vs |Return|   : r = {va[0]:.3f}, p = {va[1]:.2e}")
a1 = stats.pearsonr(ret.values[1:], ret.values[:-1])
print(f"Lag-1 autocorr Return: r = {a1[0]:.3f}, p = {a1[1]:.2e}")
a2 = stats.pearsonr(np.abs(ret.values[1:]), np.abs(ret.values[:-1]))
print(f"Lag-1 autocorr |Ret| : r = {a2[0]:.3f}, p = {a2[1]:.2e}")

### 6.b Hypothesis tests (α = 0.05)

In [ ]:
t1 = stats.ttest_1samp(ret, 0)
print(f"One-sample t-test (mean return = 0): t = {t1.statistic:.2f}, p = {t1.pvalue:.4f}")
jb = stats.jarque_bera(ret)
print(f"Jarque-Bera normality: JB = {jb.statistic:.1f}, p = {jb.pvalue:.2e}")

by_year = [g.Return.dropna().values for _, g in df.groupby("Year")]
f, p = stats.f_oneway(*by_year)
print(f"ANOVA across years   : F = {f:.2f}, p = {p:.3f}")
kw = stats.kruskal(*by_year)
print(f"Kruskal-Wallis years : H = {kw.statistic:.2f}, p = {kw.pvalue:.3f}")

wd = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
f, p = stats.f_oneway(*[df[df.Weekday == d].Return.dropna().values for d in wd])
print(f"ANOVA across weekdays: F = {f:.2f}, p = {p:.3f}")
print(df.groupby("Weekday").Return.mean().reindex(wd).round(3).to_string())

by_month = [g.Return.dropna().values for _, g in df.groupby(df.index.month)]
f, p = stats.f_oneway(*by_month)
print(f"ANOVA across months  : F = {f:.2f}, p = {p:.3f}")

### 6.c Moving-average regime (50-day vs 200-day) and volume

In [ ]:
df["Regime"] = np.where(df.MA50 > df.MA200, "Bullish (MA50 > MA200)", "Bearish (MA50 < MA200)")
df.loc[df.MA200.isna(), "Regime"] = np.nan
df["PrevRegime"] = df.Regime.shift(1)          # use yesterday's regime to avoid look-ahead
reg = df.dropna(subset=["PrevRegime", "Return"])
print(reg.groupby("PrevRegime").Return.agg(["count", "mean", "std"]).round(3))
bull = reg[reg.PrevRegime.str.startswith("Bull")].Return
bear = reg[reg.PrevRegime.str.startswith("Bear")].Return
t, p = stats.ttest_ind(bull, bear, equal_var=False)
print(f"Welch t-test: t = {t:.2f}, p = {p:.3f}")

sign = np.sign(df.MA50 - df.MA200).diff()
cross = sign[(sign != 0) & df.MA200.notna() & sign.notna()]
print("Crossovers:", [(str(d.date()), "Golden" if v > 0 else "Death") for d, v in cross.items()])

hv = df[df.Volume > df.Volume.quantile(.9)].Return.abs().mean()
lv = df[df.Volume < df.Volume.quantile(.1)].Return.abs().mean()
print(f"Mean |Return| on top-10% volume days: {hv:.2f}% vs bottom-10%: {lv:.2f}%")

## 7. Key insights
Write your own interpretation here after running the notebook (strongest finding, what it means, and one limitation).

*This analysis is for learning purposes only and is not investment advice.*